In [1]:
import autogen
autogen.__version__

'0.7.3'

In [2]:
#Dowmload ollama
#ollama pull llama3 -- Downloading purpose
#ollama run llama3 -- Prompting
#!pip install pyautogen[ollama]~(0.4.7)

In [3]:
#!pip install "pyautogen[ollama]==0.4.7"

In [4]:
#import agentops
#agentops.init("yourkey")

In [5]:
from pathlib import Path

from autogen import ConversableAgent
from autogen.coding import  CodeBlock, LocalCommandLineCodeExecutor
work_dir = Path("coding")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

#Agent 1



In [6]:
config_list = [
    {
        "model": "llama3",
        "api_type": "ollama",
        "base_url": "http://localhost:11434",
    }
]

In [7]:
# The code writer agent's system message is to instruct the LLM on how to
# use the Jupyter code executor with IPython kernel.
code_writer_system_message = """
You have been given coding capability to solve tasks using Python code.
In the following cases, suggest python code (in a python coding block) or shell script (in a sh coding block) for the user to execute.
    1. When you need to collect info, use the code to output the info you need, for example, browse or search the web, download/read a file, 
    print the content of a webpage or a file, get the current date/time, check the operating system. After sufficient info is printed and the 
    task is ready to be solved based on your language skill, you can solve the task by yourself.
    2. When you need to perform some task with code, use the code to perform the task and output the result. Finish the task smartly.
Solve the task step by step if you need to. If a plan is not provided, explain your plan first. Be clear which step uses code, and which step uses your 
language skill.When using code, you must indicate the script type in the code block. The user cannot provide any other feedback or perform any other 
action beyond executing the code you suggest. The user can't modify your code. So do not suggest incomplete code which requires users to modify. 
Don't use a code block if it's not intended to be executed by the user.
If you want the user to save the code in a file before executing it, put # filename: <filename> inside the code block as the first line. Don't include multiple code blocks in one response. Do not ask users to copy and paste the result. Instead, use 'print' function for the output when relevant. Check the execution result returned by the user.
"""

In [8]:
import os

#Agent 2

code_writer_agent = ConversableAgent(
    "code_writer",
    system_message=code_writer_system_message,
    llm_config={"config_list":config_list},
    code_execution_config=False,  # Turn off code execution for this agent.
    max_consecutive_auto_reply=2,
    human_input_mode="NEVER",
)
#Agent 1
code_executor_agent = ConversableAgent(
    "code_executor",
    system_message="Execute the Python code provided by the code writer and return the execution result.",
    llm_config={"config_list": config_list},
    code_execution_config={
        "work_dir": "coding",
        "use_docker": False
    },
    human_input_mode="NEVER",
)

In [9]:
import pprint

chat_result = code_executor_agent.initiate_chat(
    code_writer_agent,
    message="""
Write optimized Python code to calculate the number of unique permutations
of the word 'ALGEBRA'.

Requirements:
1. Use the formula:
   n! / (frequency1! * frequency2! * ...)
2. Do NOT use ASCII values or ord().
3. Verify the result using 'ALGEBRA'.
4. The expected result is 2520.
5. Include all required imports.
6. Make sure the code executes without NameError or missing imports.
7. The code should print ONLY the final permutation count.
"""
)

pprint.pprint(chat_result)

code_executor (to code_writer):


Write optimized Python code to calculate the number of unique permutations
of the word 'ALGEBRA'.

Requirements:
1. Use the formula:
   n! / (frequency1! * frequency2! * ...)
2. Do NOT use ASCII values or ord().
3. Verify the result using 'ALGEBRA'.
4. The expected result is 2520.
5. Include all required imports.
6. Make sure the code executes without NameError or missing imports.
7. The code should print ONLY the final permutation count.


--------------------------------------------------------------------------------
code_writer (to code_executor):

Here is the optimized Python code:

```python
from math import factorial

def count_permutations(word):
    freq = {}
    for char in word:
        if char in freq:
            freq[char] += 1
        else:
            freq[char] = 1
    permutation_count = factorial(len(word))
    for count in freq.values():
        permutation_count //= factorial(count)
    return permutation_count

print(count_permuta

In [10]:
dir(chat_result)


['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'chat_history',
 'chat_id',
 'cost',
 'human_input',
 'summary']

In [11]:
chat_result.chat_history

[{'content': "\nWrite optimized Python code to calculate the number of unique permutations\nof the word 'ALGEBRA'.\n\nRequirements:\n1. Use the formula:\n   n! / (frequency1! * frequency2! * ...)\n2. Do NOT use ASCII values or ord().\n3. Verify the result using 'ALGEBRA'.\n4. The expected result is 2520.\n5. Include all required imports.\n6. Make sure the code executes without NameError or missing imports.\n7. The code should print ONLY the final permutation count.\n",
  'role': 'assistant',
  'name': 'code_executor'},
 {'content': "Here is the optimized Python code:\n\n```python\nfrom math import factorial\n\ndef count_permutations(word):\n    freq = {}\n    for char in word:\n        if char in freq:\n            freq[char] += 1\n        else:\n            freq[char] = 1\n    permutation_count = factorial(len(word))\n    for count in freq.values():\n        permutation_count //= factorial(count)\n    return permutation_count\n\nprint(count_permutations('ALGEBRA'))\n```\n\nThis code f

In [12]:
chat_result.chat_id

In [13]:
chat_result.cost

{'usage_including_cached_inference': {'total_cost': 0,
  'llama3': {'cost': 0,
   'prompt_tokens': 1527,
   'completion_tokens': 281,
   'total_tokens': 1808}},
 'usage_excluding_cached_inference': {'total_cost': 0}}

In [14]:
chat_result.human_input

[]

In [15]:
chat_result.summary

"I completely agree with you. The provided Python code successfully calculates the number of unique permutations of the word 'ALGEBRA' and prints the result, which is 2520. The code is well-written, efficient, and easy to understand, making it a great example of how to solve this type of problem in Python."